In [ ]:
import os
import shutil
import pandas as pd
import numpy as np
import librosa
import soundfile as sf
from tqdm import tqdm
from pathlib import Path
from collections import defaultdict
import traceback

# ============================================================
# CONFIGURATION
# ============================================================

INPUT_ROOT = r"D:\ICASSP'27\Raw data"

OUTPUT_ROOT = r"D:\ICASSP'27\processed_data_1"

TARGET_SR = 16000

AUDIO_EXTENSIONS = (
    ".wav",
    ".flac",
    ".mp3",
    ".m4a",
    ".aac",
    ".ogg",
    ".opus",
    ".wma",
    ".aiff",
    ".aif",
    ".au"
)

# ============================================================
# OUTPUT DIRECTORIES
# ============================================================

TRAIN_OUT = os.path.join(OUTPUT_ROOT, "train")
DEV_OUT = os.path.join(OUTPUT_ROOT, "dev")
TEST_OUT = os.path.join(OUTPUT_ROOT, "eval")

os.makedirs(TRAIN_OUT, exist_ok=True)
os.makedirs(DEV_OUT, exist_ok=True)
os.makedirs(TEST_OUT, exist_ok=True)

# ============================================================
# GLOBAL STORAGE
# ============================================================

metadata_train = []
metadata_dev = []
metadata_test = []

failed_files = []

language_statistics = defaultdict(lambda: {"files": 0, "duration": 0})

model_statistics = defaultdict(lambda: {"files": 0, "duration": 0})

dataset_statistics = defaultdict(lambda: {
    "files": 0,
    "duration": 0,
    "min_duration": 999999,
    "max_duration": 0
})

duration_bins = [
    (0,2),
    (2,5),
    (5,10),
    (10,20),
    (20,30),
    (30,999999)
]

duration_statistics = defaultdict(lambda: defaultdict(int))

# ============================================================
# HELPERS
# ============================================================

def get_output_folder(split):

    if split.lower()=="train":
        return TRAIN_OUT

    if split.lower()=="dev":
        return DEV_OUT

    if split.lower()=="eval":
        return TEST_OUT

    raise ValueError("Unknown split")


def get_metadata(split):

    if split.lower()=="train":
        return metadata_train

    if split.lower()=="dev":
        return metadata_dev

    if split.lower()=="eval":
        return metadata_test

    raise ValueError("Unknown split")


def get_new_filename(counter):

    return f"fake_{counter:07d}.flac"


def duration_bucket(duration):

    for low,high in duration_bins:

        if low<=duration<high:

            if high==999999:
                return ">30 sec"

            return f"{low}-{high} sec"

    return "Unknown"

# ============================================================
# AUDIO CONVERSION
# ============================================================

def process_audio(audio_path,
                  split,
                  language,
                  model,
                  counter):

    try:

        y,sr=librosa.load(audio_path,
                          sr=None,
                          mono=False)

        original_sr=sr

        if y.ndim==1:

            channels=1

        else:

            channels=y.shape[0]
            y=np.mean(y,axis=0)

        duration=len(y)/original_sr

        y=librosa.resample(
            y,
            orig_sr=original_sr,
            target_sr=TARGET_SR
        )

        new_filename=get_new_filename(counter)

        output_folder=get_output_folder(split)

        output_path=os.path.join(
            output_folder,
            new_filename
        )

        sf.write(
            output_path,
            y,
            TARGET_SR,
            format="FLAC"
        )

        metadata=get_metadata(split)

        metadata.append({

            "filename":new_filename,

            "original_filename":os.path.basename(audio_path),

            "split":"dev" if split=="dev" else split,

            "label":"fake",

            "language":language,

            "model":model,

            "original_sampling_rate":original_sr,

            "target_sampling_rate":TARGET_SR,

            "channels":channels,

            "duration_sec":round(duration,3),

            "num_samples":len(y),

            "original_path":audio_path,

            "converted_path":output_path

        })

        language_statistics[language]["files"]+=1
        language_statistics[language]["duration"]+=duration

        model_statistics[model]["files"]+=1
        model_statistics[model]["duration"]+=duration

        split_name="dev" if split=="dev" else split

        dataset_statistics[split_name]["files"]+=1
        dataset_statistics[split_name]["duration"]+=duration

        dataset_statistics[split_name]["min_duration"]=min(
            dataset_statistics[split_name]["min_duration"],
            duration
        )

        dataset_statistics[split_name]["max_duration"]=max(
            dataset_statistics[split_name]["max_duration"],
            duration
        )

        bucket=duration_bucket(duration)

        duration_statistics[split_name][bucket]+=1

    except Exception:

        failed_files.append(audio_path)

        traceback.print_exc()

# ============================================================
# DATASET TRAVERSAL
# ============================================================

def scan_dataset():

    counters={

        "train":1,
        "dev":1,
        "eval":1

    }

    for split in ["train","dev","eval"]:

        split_path=os.path.join(
            INPUT_ROOT,
            split,
            "fake"
        )

        if not os.path.exists(split_path):

            print(f"Missing {split_path}")

            continue

        print("\n")
        print("="*60)
        print(split.upper())
        print("="*60)

        language_dirs=sorted(os.listdir(split_path))

        for language in language_dirs:

            language_path=os.path.join(
                split_path,
                language
            )

            if not os.path.isdir(language_path):

                continue

            model_dirs=sorted(
                os.listdir(language_path)
            )

            for model in model_dirs:

                model_path=os.path.join(
                    language_path,
                    model
                )

                if not os.path.isdir(model_path):

                    continue

                audio_files=[]

                for root,dirs,files in os.walk(model_path):

                    for file in files:

                        ext=os.path.splitext(file)[1].lower()

                        if ext in AUDIO_EXTENSIONS:

                            audio_files.append(
                                os.path.join(root,file)
                            )

                for audio in tqdm(
                        audio_files,
                        desc=f"{language}/{model}"):

                    process_audio(
                        audio,
                        split,
                        language,
                        model,
                        counters[split]
                    )

                    counters[split]+=1

# ============================================================
# METADATA CSV
# ============================================================

def save_metadata():

    pd.DataFrame(metadata_train).to_csv(
        os.path.join(
            OUTPUT_ROOT,
            "train_metadata.csv"
        ),
        index=False
    )

    pd.DataFrame(metadata_dev).to_csv(
        os.path.join(
            OUTPUT_ROOT,
            "dev_metadata.csv"
        ),
        index=False
    )

    pd.DataFrame(metadata_test).to_csv(
        os.path.join(
            OUTPUT_ROOT,
            "eval_metadata.csv"
        ),
        index=False
    )

In [ ]:
# ============================================================
# LANGUAGE STATISTICS
# ============================================================

def save_language_statistics():

    rows=[]

    for language in sorted(language_statistics.keys()):

        files=language_statistics[language]["files"]

        duration=language_statistics[language]["duration"]

        rows.append({

            "language":language,

            "files":files,

            "hours":round(duration/3600,3),

            "minutes":round(duration/60,2),

            "seconds":round(duration,2),

            "average_duration_sec":round(duration/files,3) if files>0 else 0

        })

    pd.DataFrame(rows).to_csv(
        os.path.join(
            OUTPUT_ROOT,
            "language_statistics.csv"
        ),
        index=False
    )

# ============================================================
# MODEL STATISTICS
# ============================================================

def save_model_statistics():

    rows=[]

    for model in sorted(model_statistics.keys()):

        files=model_statistics[model]["files"]

        duration=model_statistics[model]["duration"]

        rows.append({

            "model":model,

            "files":files,

            "hours":round(duration/3600,3),

            "minutes":round(duration/60,2),

            "seconds":round(duration,2),

            "average_duration_sec":round(duration/files,3) if files>0 else 0

        })

    pd.DataFrame(rows).to_csv(
        os.path.join(
            OUTPUT_ROOT,
            "model_statistics.csv"
        ),
        index=False
    )

# ============================================================
# DATASET STATISTICS
# ============================================================

def save_dataset_statistics():

    rows=[]

    total_files=0

    total_duration=0

    global_min=999999

    global_max=0

    for split in ["train","dev","eval"]:

        files=dataset_statistics[split]["files"]

        duration=dataset_statistics[split]["duration"]

        min_duration=dataset_statistics[split]["min_duration"]

        max_duration=dataset_statistics[split]["max_duration"]

        total_files+=files

        total_duration+=duration

        global_min=min(global_min,min_duration)

        global_max=max(global_max,max_duration)

        rows.append({

            "split":split,

            "files":files,

            "hours":round(duration/3600,3),

            "minutes":round(duration/60,2),

            "average_duration_sec":round(duration/files,3) if files>0 else 0,

            "minimum_duration_sec":round(min_duration,3),

            "maximum_duration_sec":round(max_duration,3)

        })

    rows.append({

        "split":"TOTAL",

        "files":total_files,

        "hours":round(total_duration/3600,3),

        "minutes":round(total_duration/60,2),

        "average_duration_sec":round(total_duration/total_files,3),

        "minimum_duration_sec":round(global_min,3),

        "maximum_duration_sec":round(global_max,3)

    })

    pd.DataFrame(rows).to_csv(

        os.path.join(
            OUTPUT_ROOT,
            "dataset_statistics.csv"
        ),

        index=False

    )

# ============================================================
# DURATION HISTOGRAM
# ============================================================

def save_duration_statistics():

    buckets=[

        "0-2 sec",

        "2-5 sec",

        "5-10 sec",

        "10-20 sec",

        "20-30 sec",

        ">30 sec"

    ]

    rows=[]

    for bucket in buckets:

        train=duration_statistics["train"][bucket]

        dev=duration_statistics["dev"][bucket]

        test=duration_statistics["eval"][bucket]

        rows.append({

            "duration_range":bucket,

            "train":train,

            "dev":dev,

            "eval":eval,

            "total":train+dev+eval

        })

    pd.DataFrame(rows).to_csv(

        os.path.join(
            OUTPUT_ROOT,
            "duration_statistics.csv"
        ),

        index=False

    )

# ============================================================
# FAILED FILES
# ============================================================

def save_failed_files():

    with open(

        os.path.join(
            OUTPUT_ROOT,
            "failed_files.txt"
        ),

        "w",

        encoding="utf-8"

    ) as f:

        for file in failed_files:

            f.write(file+"\n")

# ============================================================
# PRINT SUMMARY
# ============================================================

def print_summary():

    total_files=0

    total_duration=0

    total_languages=len(language_statistics)

    total_models=len(model_statistics)

    print()

    print("="*70)

    print("DATASET SUMMARY")

    print("="*70)

    for split in ["train","dev","eval"]:

        files=dataset_statistics[split]["files"]

        duration=dataset_statistics[split]["duration"]

        total_files+=files

        total_duration+=duration

        print(f"{split.upper():8s}: {files:8d} files   {duration/3600:.2f} hrs")

    print()

    print(f"TOTAL FILES        : {total_files}")

    print(f"TOTAL HOURS        : {total_duration/3600:.3f}")

    print(f"TOTAL LANGUAGES    : {total_languages}")

    print(f"TOTAL MODELS       : {total_models}")

    print(f"FAILED FILES       : {len(failed_files)}")

    print()

    print("Original languages")

    for language in sorted(language_statistics):

        print(

            f"{language:20s}"

            f"{language_statistics[language]['files']:8d} files"

            f"{language_statistics[language]['duration']/3600:8.2f} hrs"

        )

    print()

    print("Models")

    for model in sorted(model_statistics):

        print(

            f"{model:20s}"

            f"{model_statistics[model]['files']:8d} files"

            f"{model_statistics[model]['duration']/3600:8.2f} hrs"

        )

    print("="*70)

# ============================================================
# MAIN
# ============================================================

def main():

    print()

    print("="*70)

    print("STARTING DATASET CONVERSION")

    print("="*70)

    scan_dataset()

    print()

    print("Saving metadata...")

    save_metadata()

    print("Saving language statistics...")

    save_language_statistics()

    print("Saving model statistics...")

    save_model_statistics()

    print("Saving duration statistics...")

    save_duration_statistics()

    print("Saving dataset statistics...")

    save_dataset_statistics()

    print("Saving failed file list...")

    save_failed_files()

    print_summary()

    print()

    print("="*70)

    print("FINISHED SUCCESSFULLY")

    print("="*70)

# ============================================================
# ENTRY
# ============================================================

if __name__=="__main__":

    main()


STARTING DATASET CONVERSION


TRAIN


ar/tts_models_multilingual_multi-dataset_xtts_v1.1:   0%|          | 0/300 [00:00<?, ?it/s]c:\Users\admin\anaconda3\envs\speech\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
ar/tts_models_multilingual_multi-dataset_xtts_v1.1: 100%|██████████| 300/300 [00:09<00:00, 32.88it/s]
de/tts_models_de_thorsten_tacotron2-DDC: 100%|██████████| 300/300 [00:04<00:00, 71.70it/s]
de/tts_models_multilingual_multi-dataset_xtts_v1.1: 100%|██████████| 300/300 [00:05<00:00, 59.85it/s]
en/tts_models_en_ljspeech_vits--neon: 100%|██████████| 300/300 [00:05<00:00, 54.69it/s]
en/tts_models_multilingual_multi-dataset_xtts_v1.1: 100%|██████████| 300/300 [00:05<00:00, 57.36it/s]
fr/tts_models_fr_mai_tacotron2-DDC: 100%|██████████| 300/300 [00:04<00:00, 66.53it/s]
fr/tts_models_multilingual_multi-dataset_xtts_v1.1: 100%|██████████| 300/300 [



DEV


uk/tts_models_uk_mai_vits: 100%|██████████| 300/300 [00:04<00:00, 74.62it/s]
zh-cn/tts_models_multilingual_multi-dataset_xtts_v1.1: 100%|██████████| 300/300 [00:04<00:00, 60.14it/s]
zh-cn/tts_models_multilingual_multi-dataset_xtts_v2: 100%|██████████| 300/300 [00:04<00:00, 64.99it/s]




EVAL


ar/tts_models_multilingual_multi-dataset_xtts_v1.1: 100%|██████████| 300/300 [00:05<00:00, 52.62it/s]
cs/tts_models_multilingual_multi-dataset_bark: 100%|██████████| 300/300 [00:06<00:00, 46.02it/s]
cs/tts_models_multilingual_multi-dataset_xtts_v1.1: 100%|██████████| 300/300 [00:05<00:00, 58.15it/s]
de/tts_models_multilingual_multi-dataset_bark: 100%|██████████| 300/300 [00:05<00:00, 59.42it/s]
de/tts_models_multilingual_multi-dataset_xtts_v1.1: 100%|██████████| 300/300 [00:04<00:00, 65.68it/s]
en/tts_models_multilingual_multi-dataset_bark: 100%|██████████| 300/300 [00:05<00:00, 55.90it/s]
en/tts_models_multilingual_multi-dataset_xtts_v1.1: 100%|██████████| 300/300 [00:04<00:00, 66.19it/s]
es/tts_models_multilingual_multi-dataset_bark: 100%|██████████| 300/300 [00:05<00:00, 58.76it/s]
es/tts_models_multilingual_multi-dataset_xtts_v1.1: 100%|██████████| 300/300 [00:04<00:00, 63.87it/s]
fr/tts_models_fr_mai_tacotron2-DDC: 100%|██████████| 300/300 [00:03<00:00, 76.94it/s]
fr/tts_models_mu


Saving metadata...
Saving language statistics...
Saving model statistics...
Saving duration statistics...
Saving dataset statistics...
Saving failed file list...

DATASET SUMMARY
TRAIN   :    11100 files   23.33 hrs
DEV     :    12000 files   28.48 hrs
TEST    :        0 files   0.00 hrs

TOTAL FILES        : 23100
TOTAL HOURS        : 51.806
TOTAL LANGUAGES    : 38
TOTAL MODELS       : 82
FAILED FILES       : 0

Original languages
ar                      1500 files    4.60 hrs
bg                       600 files    1.81 hrs
bn                       900 files    2.85 hrs
cs                      1200 files    3.38 hrs
da                       600 files    1.71 hrs
de                      4500 files    9.21 hrs
el                       600 files    2.05 hrs
en                     14591 files   30.13 hrs
es                      2100 files    3.74 hrs
et                       600 files    1.69 hrs
fa                       600 files    1.99 hrs
fi                       900 files    2.77 hrs

In [ ]:
import pandas as pd

train_path=r"D:\ICASSP'27\processed_data_1\train_metadata.csv"
dev_path=r"D:\ICASSP'27\processed_data_1\dev_metadata.csv"
eval_path=r"D:\ICASSP'27\processed_data_1\test_metadata.csv"

train_df=pd.read_csv(train_path)
dev_df=pd.read_csv(dev_path)
eval_df=pd.read_csv(eval_path)

train_models=set(train_df["model"].astype(str).str.strip().str.lower())
train_languages=set(train_df["language"].astype(str).str.strip().str.lower())

def add_seen_columns(df):
    df["architecture_seen"]=df["model"].astype(str).str.strip().str.lower().apply(lambda x:"Seen" if x in train_models else "Unseen")
    df["language_seen"]=df["language"].astype(str).str.strip().str.lower().apply(lambda x:"Seen" if x in train_languages else "Unseen")
    return df

dev_df=add_seen_columns(dev_df)
eval_df=add_seen_columns(eval_df)

dev_df.to_excel(dev_path.replace(".csv","_seen.xlsx"),index=False)
eval_df.to_excel(eval_path.replace(".csv","_seen.xlsx"),index=False)

print("="*70)
print("TRAINING INFORMATION")
print("="*70)
print(f"Number of training architectures : {len(train_models)}")
print(f"Architectures : {sorted(train_models)}")
print()
print(f"Number of training languages     : {len(train_languages)}")
print(f"Languages : {sorted(train_languages)}")

for name,df in [("DEV",dev_df),("TEST",eval_df)]:
    print("\n"+"="*70)
    print(name+" STATISTICS")
    print("="*70)

    print("\nArchitecture Seen/Unseen")
    print(df["architecture_seen"].value_counts())

    print("\nLanguage Seen/Unseen")
    print(df["language_seen"].value_counts())

    unseen_models=sorted(df.loc[df["architecture_seen"]=="Unseen","model"].astype(str).str.strip().str.lower().unique())
    unseen_languages=sorted(df.loc[df["language_seen"]=="Unseen","language"].astype(str).str.strip().str.lower().unique())

    print("\nUnseen Architectures:")
    if len(unseen_models):
        for m in unseen_models:
            print("  ",m)
    else:
        print("  None")

    print("\nUnseen Languages:")
    if len(unseen_languages):
        for l in unseen_languages:
            print("  ",l)
    else:
        print("  None")

    print("\nArchitecture Distribution")
    print(pd.crosstab(df["model"],df["architecture_seen"]))

    print("\nLanguage Distribution")
    print(pd.crosstab(df["language"],df["language_seen"]))

print("\nFinished.")

TRAINING INFORMATION
Number of training architectures : 24
Architectures : ['facebook_mms-tts-deu', 'griffin_lim', 'mars5', 'melotts', 'metavoice-1b', 'suno_bark', 'suno_bark-small', 'tts_models_de_css10_vits-neon', 'tts_models_de_thorsten_tacotron2-ddc', 'tts_models_en_ljspeech_fast_pitch', 'tts_models_en_ljspeech_speedy-speech', 'tts_models_en_ljspeech_tacotron2-dca', 'tts_models_en_ljspeech_tacotron2-ddc', 'tts_models_en_ljspeech_tacotron2-ddc_ph', 'tts_models_en_ljspeech_vits--neon', 'tts_models_fr_css10_vits', 'tts_models_fr_mai_tacotron2-ddc', 'tts_models_it_mai_female_glow-tts', 'tts_models_it_mai_female_vits', 'tts_models_it_mai_male_vits', 'tts_models_lt_cv_vits', 'tts_models_multilingual_multi-dataset_xtts_v1.1', 'tts_models_multilingual_multi-dataset_xtts_v2', 'vixtts']

Number of training languages     : 8
Languages : ['ar', 'de', 'en', 'fr', 'it', 'ko', 'lt', 'pl']

DEV STATISTICS

Architecture Seen/Unseen
architecture_seen
Unseen    7200
Seen      4800
Name: count, dtype: